# FedKG-Maint — Phase 3 Local-Normalization Closure Audit v1

## Purpose and scope

This is a post-run audit of `FedKG_Maint_Phase3_Privacy_Utility_LocalNorm_v1.ipynb`. It does not retrain models, tune hyperparameters, or select a configuration from test results.

It will:

1. Verify that the selected run declares `policy="local"` and that the saved final results record the same policy.
2. Recompute PR-AUC from the saved predictions and compare it with the local-normalization result files.
3. Recompute the scoped client-update privacy budget using manual RDP conversion and Google's `dp-accounting` library when available.
4. Run a paired machine-cluster bootstrap comparing `DP-FedAvg-CompactLinear` with `ClippedFedAvg`.
5. Write machine-readable and Markdown audit reports to `phase3_closure_audit_localnorm_v1/`.

**Important:** `test_site_full` was used in earlier project experiments. This is an integrity and uncertainty audit on the existing benchmark, not a new untouched external evaluation. Do not use these results to select another model or tune hyperparameters.

The factory partitions and training are still simulated within one Python process. Local scalers align the preprocessing protocol with the project requirement, but do not by themselves prove physical raw-data isolation.

## Local-normalization run results supplied before audit

The completed run reported:

- DP-FedAvg mean test PR-AUC: `0.3186` (SD `0.0301`).
- ClippedFedAvg mean test PR-AUC: `0.2875` (SD `0.0022`).
- Central Compact Linear mean test PR-AUC: `0.3702` (SD `0.0009`).
- Selected settings: `clip_norm=0.1`, `noise_multiplier=1.0`, `local_epochs=1`, `rounds=3`.
- Manual update-channel privacy bound on the original RDP grid: `epsilon=9.837642`, `delta=1e-5`.

This audit recomputes those values from the files in Google Drive and additionally calculates a paired machine-bootstrap confidence interval for the local-normalization experiment. No confidence interval from the earlier global-normalization audit is reused here.


## 1. Paths and prerequisite checks

In [ ]:

import os, sys, json, math, subprocess, warnings, importlib
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/FedKG-Maint")
except ImportError:
    PROJECT_DIR = Path(os.environ.get("FEDKG_ROOT", "./FedKG-Maint")).resolve()

PREP_DIR = PROJECT_DIR / "prep_v3"
LOCAL_DIR = PROJECT_DIR / "phase3_federated_privacy_localnorm_v1"
AUDIT_DIR = PROJECT_DIR / "phase3_closure_audit_localnorm_v1"
for sub in ["results", "reports", "logs"]:
    (AUDIT_DIR / sub).mkdir(parents=True, exist_ok=True)

SEEDS = [42, 43, 44]
METHODS = ["DP-FedAvg-CompactLinear", "ClippedFedAvg", "FedAvg", "CentralCompactLinear"]
PRIMARY_DP = "DP-FedAvg-CompactLinear"
PRIMARY_CONTROL = "ClippedFedAvg"
DELTA = 1e-5
EPSILON_CAP = 10.0
BOOTSTRAP_B = 2000
BOOTSTRAP_RANDOM_SEED = 20261018

required = [
    PREP_DIR / "fedkg_data.py",
    LOCAL_DIR / "reports" / "compact_dp_selection_lock.json",
    LOCAL_DIR / "results" / "compact_dp_final_test_results.csv",
    LOCAL_DIR / "results" / "compact_dp_final_summary.csv",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Required Phase 1 v3 or local-normalization run outputs are missing:\n" + "\n".join(missing))

PRED_DIR = LOCAL_DIR / "predictions"
missing_predictions = [
    str(PRED_DIR / f"{method}__seed{seed}_test_site_full.parquet")
    for method in METHODS for seed in SEEDS
    if not (PRED_DIR / f"{method}__seed{seed}_test_site_full.parquet").exists()
]
if missing_predictions:
    raise FileNotFoundError("Missing saved local-normalization predictions:\n" + "\n".join(missing_predictions))

lock = json.loads((LOCAL_DIR / "reports" / "compact_dp_selection_lock.json").read_text())
final_results = pd.read_csv(LOCAL_DIR / "results" / "compact_dp_final_test_results.csv")
final_summary_saved = pd.read_csv(LOCAL_DIR / "results" / "compact_dp_final_summary.csv")

NORMALIZATION_POLICY = str(lock.get("normalization_policy", ""))
ROUNDS = int(lock["rounds"])
selected_sigma = float(lock["selected_noise_multiplier"])
selected_clip = float(lock["selected_clip_norm"])
selected_local_epochs = int(lock["selected_local_epochs"])
lock_epsilon = float(lock["selected_epsilon"])

print("Project:", PROJECT_DIR)
print("Local-normalization run:", LOCAL_DIR)
print("Audit output:", AUDIT_DIR)
print("Found prediction files:", len(METHODS) * len(SEEDS))
print("Normalization policy in lock:", NORMALIZATION_POLICY)
print("Selected settings:", {
    "local_epochs": selected_local_epochs,
    "clip_norm": selected_clip,
    "noise_multiplier": selected_sigma,
    "rounds": ROUNDS,
    "epsilon_from_lock": lock_epsilon,
})


## 2. Independent privacy-accounting cross-check

The manual and library conversion methods may report different epsilon values. Numerical equality is not required. The audit checks that the values are finite, satisfy the predeclared `epsilon <= 10` cap, and that the library values are no larger than the corresponding manual bounds within tolerance for this configuration.

The privacy claim remains limited to the clipped/noised client-update channel under the recorded assumptions: add/remove-one-client adjacency, trusted server, fixed denominator, full participation, and no subsampling amplification. It is not an end-to-end differential-privacy claim.


In [ ]:

DP_ACCOUNTING_AVAILABLE = False
DP_ACCOUNTING_INITIAL_IMPORT_ERROR = None
DP_ACCOUNTING_INSTALL_ERROR = None
DP_ACCOUNTING_INSTALL_ATTEMPTED = False

try:
    from dp_accounting import dp_event, privacy_accountant
    from dp_accounting.rdp import rdp_privacy_accountant
    DP_ACCOUNTING_AVAILABLE = True
except Exception as e:
    DP_ACCOUNTING_INITIAL_IMPORT_ERROR = repr(e)
    DP_ACCOUNTING_INSTALL_ATTEMPTED = True
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "dp-accounting==0.6.0"],
            check=True, timeout=180
        )
        from dp_accounting import dp_event, privacy_accountant
        from dp_accounting.rdp import rdp_privacy_accountant
        DP_ACCOUNTING_AVAILABLE = True
    except Exception as e2:
        DP_ACCOUNTING_INSTALL_ERROR = repr(e2)

if DP_ACCOUNTING_AVAILABLE:
    if DP_ACCOUNTING_INITIAL_IMPORT_ERROR is None:
        DP_ACCOUNTING_STATUS = "available_on_initial_import"
    elif DP_ACCOUNTING_INSTALL_ERROR is None:
        DP_ACCOUNTING_STATUS = "installed_successfully_after_initial_import_failure"
    else:
        DP_ACCOUNTING_STATUS = "available_after_retry"
else:
    DP_ACCOUNTING_STATUS = "unavailable_after_install_attempt" if DP_ACCOUNTING_INSTALL_ATTEMPTED else "unavailable"

try:
    from importlib.metadata import version as package_version
    DP_ACCOUNTING_VERSION = package_version("dp-accounting") if DP_ACCOUNTING_AVAILABLE else None
except Exception:
    DP_ACCOUNTING_VERSION = "unknown" if DP_ACCOUNTING_AVAILABLE else None

V3_ORDERS = np.array([1.25, 1.5, 2, 3, 4, 5, 8, 10, 16, 32, 64, 128, 256, 512], dtype=float)
DENSE_ORDERS = np.unique(np.concatenate([
    np.arange(1.01, 10.0, 0.01),
    np.arange(10.0, 100.0, 0.1),
    np.array([128.0, 256.0, 512.0, 1024.0])
]))

def manual_rdp_epsilon(noise_multiplier, steps, delta, orders):
    rdp = steps * orders / (2.0 * noise_multiplier ** 2)
    values = rdp + math.log(1.0 / delta) / (orders - 1.0)
    idx = int(np.argmin(values))
    return float(values[idx]), float(orders[idx])

def library_rdp_epsilon(noise_multiplier, steps, delta, orders):
    accountant = rdp_privacy_accountant.RdpAccountant(
        orders=list(map(float, orders)),
        neighboring_relation=privacy_accountant.NeighboringRelation.ADD_OR_REMOVE_ONE
    )
    accountant.compose(dp_event.SelfComposedDpEvent(
        dp_event.GaussianDpEvent(float(noise_multiplier)), int(steps)
    ))
    return float(accountant.get_epsilon(float(delta)))

manual_v3_eps, manual_v3_order = manual_rdp_epsilon(selected_sigma, ROUNDS, DELTA, V3_ORDERS)
manual_dense_eps, manual_dense_order = manual_rdp_epsilon(selected_sigma, ROUNDS, DELTA, DENSE_ORDERS)

if DP_ACCOUNTING_AVAILABLE:
    library_v3_eps = library_rdp_epsilon(selected_sigma, ROUNDS, DELTA, V3_ORDERS)
    library_dense_eps = library_rdp_epsilon(selected_sigma, ROUNDS, DELTA, DENSE_ORDERS)
else:
    library_v3_eps = np.nan
    library_dense_eps = np.nan

privacy_audit = {
    "normalization_policy": NORMALIZATION_POLICY,
    "selected_noise_multiplier": selected_sigma,
    "selected_clip_norm": selected_clip,
    "selected_local_epochs": selected_local_epochs,
    "rounds": ROUNDS,
    "delta": DELTA,
    "epsilon_cap": EPSILON_CAP,
    "epsilon_stored_in_selection_lock": lock_epsilon,
    "epsilon_recomputed_v3_order_grid": manual_v3_eps,
    "optimal_order_v3_grid": manual_v3_order,
    "epsilon_recomputed_dense_order_grid": manual_dense_eps,
    "optimal_order_dense_grid": manual_dense_order,
    "dp_accounting_package_available": DP_ACCOUNTING_AVAILABLE,
    "dp_accounting_status": DP_ACCOUNTING_STATUS,
    "dp_accounting_package_version": DP_ACCOUNTING_VERSION,
    "dp_accounting_initial_import_error": DP_ACCOUNTING_INITIAL_IMPORT_ERROR,
    "dp_accounting_install_error": DP_ACCOUNTING_INSTALL_ERROR,
    "epsilon_dp_accounting_same_grid": library_v3_eps,
    "epsilon_dp_accounting_dense_grid": library_dense_eps,
    "privacy_scope": "client-update mechanism only; add/remove-one client adjacency; trusted server; fixed denominator; full participation; no subsampling amplification"
}
print(json.dumps(privacy_audit, indent=2))

assert np.isclose(lock_epsilon, manual_v3_eps, atol=1e-8), "Stored epsilon does not reproduce from the original RDP order grid."
assert manual_dense_eps <= manual_v3_eps + 1e-8, "Dense-order manual RDP bound should not be looser than sparse-grid bound."
assert manual_v3_eps <= EPSILON_CAP + 1e-8, "Manual privacy accounting exceeds the study epsilon cap."

if DP_ACCOUNTING_AVAILABLE:
    assert np.isfinite(library_v3_eps) and library_v3_eps <= manual_v3_eps + 1e-7, (
        f"Library epsilon exceeds manual bound: {library_v3_eps} > {manual_v3_eps}"
    )
    assert np.isfinite(library_dense_eps) and library_dense_eps <= manual_dense_eps + 1e-6, (
        f"Library dense-grid epsilon exceeds manual bound: {library_dense_eps} > {manual_dense_eps}"
    )
    assert library_v3_eps <= EPSILON_CAP + 1e-7, "Library epsilon exceeds study cap."
    assert library_dense_eps <= EPSILON_CAP + 1e-6, "Dense-grid library epsilon exceeds study cap."
else:
    print("WARNING: dp-accounting was unavailable after installation attempt; manual RDP checks remain available, but independent library verification is unavailable.")


## 3. Reproduce saved predictions and verify local normalization

In [ ]:

sys.path.insert(0, str(PREP_DIR))
sys.modules.pop("fedkg_data", None)
importlib.invalidate_caches()
import fedkg_data

Xtest_all, ytest, mtest = fedkg_data.load_split(
    "test_site_full", policy="local", feature_set="full"
)
ytest = np.asarray(ytest).reshape(-1).astype(np.int8)
meta = mtest.reset_index(drop=True).copy()

assert "machineID" in meta.columns and "datetime" in meta.columns
assert "client_id" in meta.columns
assert set(meta["client_id"].astype(str).unique()) == {"factory_5"}, "Test rows must come only from factory_5."
assert NORMALIZATION_POLICY == "local", f"Selection lock declares policy={NORMALIZATION_POLICY}, not local."
assert "normalization_policy" in final_results.columns, "Results CSV lacks normalization_policy provenance."
assert set(final_results["normalization_policy"].astype(str).unique()) == {"local"}, "Results CSV includes non-local normalization."
assert str(lock.get("selection_split", "")) == "validation only", "Selection lock does not declare validation-only selection."
assert lock.get("test_loaded_before_selection") is False, "Selection lock does not declare test loading after selection."

meta_machines = meta["machineID"].astype(str).to_numpy()
meta_dates = pd.to_datetime(meta["datetime"], utc=True).astype("int64").to_numpy()
assert len(ytest) == len(meta) == len(Xtest_all)
assert np.isfinite(ytest).all()

PRED = {}
computed_rows = []
consistency_rows = []

for method in METHODS:
    for seed in SEEDS:
        path = PRED_DIR / f"{method}__seed{seed}_test_site_full.parquet"
        pred = pd.read_parquet(path)
        required_cols = {"machineID", "datetime", "score", "seed"}
        assert required_cols.issubset(pred.columns), f"Required columns missing in {path.name}"
        assert len(pred) == len(meta), f"Wrong prediction row count in {path.name}"
        assert pred["seed"].nunique() == 1 and int(pred["seed"].iloc[0]) == seed
        assert np.array_equal(pred["machineID"].astype(str).to_numpy(), meta_machines), f"Machine row order mismatch: {path.name}"
        pred_dates = pd.to_datetime(pred["datetime"], utc=True).astype("int64").to_numpy()
        assert np.array_equal(pred_dates, meta_dates), f"Timestamp row order mismatch: {path.name}"
        scores = pred["score"].to_numpy(dtype=np.float64)
        assert np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all()

        ap = float(average_precision_score(ytest, scores))
        saved = final_results[(final_results["method"] == method) & (final_results["seed"] == seed)]
        assert len(saved) == 1, f"Expected one saved row for {method}, seed {seed}"
        saved_ap = float(saved["test_site_full_AP"].iloc[0])
        consistency_rows.append({
            "method": method, "seed": seed,
            "recomputed_AP_from_saved_predictions": ap,
            "AP_in_original_results_csv": saved_ap,
            "absolute_difference": abs(ap - saved_ap)
        })
        computed_rows.append({
            "method": method, "seed": seed,
            "recomputed_test_AP": ap,
            "test_prevalence": float(ytest.mean()),
            "n_test_rows": len(ytest),
            "n_machines": int(meta["machineID"].nunique()),
            "normalization_policy": NORMALIZATION_POLICY
        })
        PRED[(method, seed)] = {"scores": scores, "path": str(path)}

AP_CHECK = pd.DataFrame(consistency_rows)
AP_TABLE = pd.DataFrame(computed_rows)
AP_CHECK.to_csv(AUDIT_DIR / "results" / "prediction_ap_reproducibility.csv", index=False)
AP_TABLE.to_csv(AUDIT_DIR / "results" / "recomputed_test_ap_by_seed.csv", index=False)

max_ap_reproduction_error = float(AP_CHECK["absolute_difference"].max())
SUMMARY_RECALC = (
    AP_TABLE.groupby("method", as_index=False)
    .agg(seeds=("seed", "nunique"),
         mean_test_AP=("recomputed_test_AP", "mean"),
         sd_test_AP=("recomputed_test_AP", "std"))
)
SUMMARY_CHECK = SUMMARY_RECALC.merge(
    final_summary_saved[["method", "seeds", "mean_test_AP", "sd_test_AP"]],
    on="method", suffixes=("_recomputed", "_saved"), validate="one_to_one"
)
SUMMARY_CHECK["mean_AP_abs_error"] = (
    SUMMARY_CHECK["mean_test_AP_recomputed"] - SUMMARY_CHECK["mean_test_AP_saved"]
).abs()
SUMMARY_CHECK["sd_AP_abs_error"] = (
    SUMMARY_CHECK["sd_test_AP_recomputed"] - SUMMARY_CHECK["sd_test_AP_saved"]
).abs()
max_summary_error = float(
    SUMMARY_CHECK[["mean_AP_abs_error", "sd_AP_abs_error"]].to_numpy().max()
)
SUMMARY_CHECK.to_csv(AUDIT_DIR / "results" / "summary_ap_reproducibility.csv", index=False)

print("Test rows:", len(ytest), "| machines:", meta["machineID"].nunique(),
      "| prevalence:", round(float(ytest.mean()), 6))
print("Held-out factory:", sorted(meta["client_id"].astype(str).unique()))
print("Normalization policy:", NORMALIZATION_POLICY)
print("Largest saved-prediction AP reproduction difference:", max_ap_reproduction_error)
print("Largest saved-summary AP/SD reproduction difference:", max_summary_error)
print("\nRecomputed AP by method and seed:")
print(AP_TABLE.pivot(index="seed", columns="method", values="recomputed_test_AP").to_string(float_format=lambda x: f"{x:.4f}"))
print("\nSummary AP cross-check:")
print(SUMMARY_CHECK.to_string(index=False, float_format=lambda x: f"{x:.6g}"))

assert max_ap_reproduction_error <= 1e-6, "Recomputed AP mismatch; investigate saved scores/data alignment."
assert max_summary_error <= 1e-6, "Recomputed summary mismatch against saved CSV."


## 4. Paired machine-cluster bootstrap

The primary comparison is `DP-FedAvg-CompactLinear` minus `ClippedFedAvg`. The same held-out rows, machine clusters, and seeds are paired across methods. Each bootstrap replicate samples the 20 held-out machines with replacement and keeps each sampled machine's time-series rows together.

The resulting interval is conditional on the trained models and this existing benchmark. It is not a formal p-value and does not establish generalization to unseen factories.


In [ ]:

cluster_codes, cluster_ids = pd.factorize(meta["machineID"].astype(str), sort=True)
n_clusters = len(cluster_ids)
assert n_clusters == 20, f"Expected 20 held-out machines; found {n_clusters}."

def make_ap_cache(y, scores, cluster_codes):
    order = np.argsort(-scores, kind="mergesort")
    sorted_scores = scores[order]
    sorted_y = y[order].astype(np.float64)
    sorted_clusters = cluster_codes[order]
    new_group = np.r_[True, sorted_scores[1:] != sorted_scores[:-1]]
    group_codes = np.cumsum(new_group, dtype=np.int64) - 1
    return {
        "groups": group_codes,
        "y": sorted_y,
        "clusters": sorted_clusters,
        "n_groups": int(group_codes[-1]) + 1
    }

def weighted_ap_from_cache(cache, cluster_weights):
    row_weights = cluster_weights[cache["clusters"]].astype(np.float64)
    group_total = np.bincount(
        cache["groups"], weights=row_weights, minlength=cache["n_groups"]
    )
    group_positive = np.bincount(
        cache["groups"], weights=row_weights * cache["y"], minlength=cache["n_groups"]
    )
    total_positive = float(group_positive.sum())
    if total_positive <= 0:
        return np.nan
    cum_total = np.cumsum(group_total)
    cum_positive = np.cumsum(group_positive)
    precision_at_group = cum_positive / np.maximum(cum_total, 1e-300)
    return float(np.sum((group_positive / total_positive) * precision_at_group))

AP_CACHE = {}
for method in METHODS:
    for seed in SEEDS:
        scores = PRED[(method, seed)]["scores"]
        cache = make_ap_cache(ytest, scores, cluster_codes)
        check_ap = weighted_ap_from_cache(cache, np.ones(n_clusters, dtype=np.float64))
        sklearn_ap = float(average_precision_score(ytest, scores))
        if not np.isclose(check_ap, sklearn_ap, atol=1e-10, rtol=1e-10):
            raise AssertionError(
                f"Weighted AP check failed for {method}, seed {seed}: {check_ap} vs {sklearn_ap}"
            )
        AP_CACHE[(method, seed)] = cache

rng = np.random.default_rng(BOOTSTRAP_RANDOM_SEED)
seed_diffs = np.full((BOOTSTRAP_B, len(SEEDS)), np.nan, dtype=np.float64)
mean_diffs = np.full(BOOTSTRAP_B, np.nan, dtype=np.float64)

for b in range(BOOTSTRAP_B):
    sampled_clusters = rng.integers(0, n_clusters, size=n_clusters)
    cluster_weights = np.bincount(sampled_clusters, minlength=n_clusters).astype(np.float64)
    for j, seed in enumerate(SEEDS):
        dp_ap = weighted_ap_from_cache(AP_CACHE[(PRIMARY_DP, seed)], cluster_weights)
        control_ap = weighted_ap_from_cache(AP_CACHE[(PRIMARY_CONTROL, seed)], cluster_weights)
        seed_diffs[b, j] = dp_ap - control_ap
    mean_diffs[b] = float(np.mean(seed_diffs[b, :]))

finite = np.isfinite(mean_diffs)
boot_diff = mean_diffs[finite]
boot_ci = np.quantile(boot_diff, [0.025, 0.975])

point_dp_aps = np.array([
    float(AP_TABLE[(AP_TABLE.method == PRIMARY_DP) & (AP_TABLE.seed == seed)]["recomputed_test_AP"].iloc[0])
    for seed in SEEDS
])
point_ctrl_aps = np.array([
    float(AP_TABLE[(AP_TABLE.method == PRIMARY_CONTROL) & (AP_TABLE.seed == seed)]["recomputed_test_AP"].iloc[0])
    for seed in SEEDS
])
point_diff_each_seed = point_dp_aps - point_ctrl_aps
point_diff_mean = float(np.mean(point_diff_each_seed))

paired_rows = []
for j, seed in enumerate(SEEDS):
    ci_seed = np.quantile(seed_diffs[np.isfinite(seed_diffs[:, j]), j], [0.025, 0.975])
    paired_rows.append({
        "comparison": f"{PRIMARY_DP} minus {PRIMARY_CONTROL}",
        "seed": seed,
        "DP_AP": point_dp_aps[j],
        "control_AP": point_ctrl_aps[j],
        "observed_AP_difference": point_diff_each_seed[j],
        "paired_machine_bootstrap_CI_low": ci_seed[0],
        "paired_machine_bootstrap_CI_high": ci_seed[1]
    })
paired_rows.append({
    "comparison": f"{PRIMARY_DP} minus {PRIMARY_CONTROL}",
    "seed": "mean_over_seeds",
    "DP_AP": float(point_dp_aps.mean()),
    "control_AP": float(point_ctrl_aps.mean()),
    "observed_AP_difference": point_diff_mean,
    "paired_machine_bootstrap_CI_low": float(boot_ci[0]),
    "paired_machine_bootstrap_CI_high": float(boot_ci[1])
})
PAIRED = pd.DataFrame(paired_rows)
PAIRED["bootstrap_replicates"] = BOOTSTRAP_B
PAIRED["bootstrap_unit"] = "machineID (20 held-out machines), paired across methods and seeds"
PAIRED.to_csv(AUDIT_DIR / "results" / "paired_machine_bootstrap_dp_vs_clipped.csv", index=False)

print("Primary paired comparison: DP - ClippedFedAvg")
print(PAIRED.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
bootstrap_nonpositive_fraction = float(np.mean(boot_diff <= 0))
print("Descriptive bootstrap fraction with mean difference <= 0:",
      round(bootstrap_nonpositive_fraction, 4), "(not a formal p-value)")
print("95% paired machine-cluster bootstrap CI:", (float(boot_ci[0]), float(boot_ci[1])))

prevalence = float(np.mean(ytest))
dp_mean = float(point_dp_aps.mean())
dp_sd = float(point_dp_aps.std(ddof=1))
control_mean = float(point_ctrl_aps.mean())
central_mean = float(AP_TABLE[AP_TABLE["method"] == "CentralCompactLinear"]["recomputed_test_AP"].mean())
utility_floor = 3.0 * prevalence
utility_screen_pass = bool(dp_mean >= utility_floor)
stability_screen_pass = bool(np.all(point_dp_aps > prevalence) and dp_sd <= max(dp_mean, 1e-12))
ci_above_zero = bool(boot_ci[0] > 0)
print("\nMean-utility screen:", utility_screen_pass,
      "| mean DP AP:", round(dp_mean, 4), "| 3x prevalence:", round(utility_floor, 4))
print("Descriptive seed-stability screen:", stability_screen_pass,
      "| seed SD:", round(dp_sd, 4), "| mean AP:", round(dp_mean, 4))
print("Paired bootstrap CI entirely above zero:", ci_above_zero)

# Informational diagnostic: the clipped baseline may match ordinary FedAvg if clipping was inactive.
baseline_prediction_differences = {}
for seed in SEEDS:
    a = PRED[("FedAvg", seed)]["scores"]
    b = PRED[("ClippedFedAvg", seed)]["scores"]
    baseline_prediction_differences[str(seed)] = float(np.max(np.abs(a - b)))
print("Max absolute score difference between FedAvg and ClippedFedAvg by seed:",
      baseline_prediction_differences)


## 5. Write reports and audit decision

In [ ]:

checks = {}

def record_check(name, ok, detail=""):
    checks[name] = {"status": "PASS" if bool(ok) else "FAIL", "detail": str(detail)}
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {detail}")

record_check(
    "LOCAL-AUDIT-1 local normalization policy declared",
    NORMALIZATION_POLICY == "local",
    f"selection-lock policy={NORMALIZATION_POLICY}"
)
record_check(
    "LOCAL-AUDIT-2 results CSV identifies local normalization",
    "normalization_policy" in final_results.columns
    and set(final_results["normalization_policy"].astype(str).unique()) == {"local"},
    f"policies={final_results['normalization_policy'].astype(str).unique().tolist() if 'normalization_policy' in final_results.columns else 'missing column'}"
)
record_check(
    "LOCAL-AUDIT-3 all 12 predictions aligned to factory_5 test rows",
    len(PRED) == 12 and set(meta["client_id"].astype(str).unique()) == {"factory_5"},
    f"prediction_files={len(PRED)}; held_out={sorted(meta['client_id'].astype(str).unique())}"
)
record_check(
    "LOCAL-AUDIT-4 saved-prediction AP reproducibility",
    max_ap_reproduction_error <= 1e-6,
    f"max_abs_error={max_ap_reproduction_error:.12g}"
)
record_check(
    "LOCAL-AUDIT-5 summary mean/SD reproducibility",
    max_summary_error <= 1e-6,
    f"max_abs_error={max_summary_error:.12g}"
)
record_check(
    "LOCAL-AUDIT-6 stored privacy epsilon reproduces",
    np.isclose(lock_epsilon, manual_v3_eps, atol=1e-8),
    f"stored={lock_epsilon:.6f}; recomputed={manual_v3_eps:.6f}"
)
record_check(
    "LOCAL-AUDIT-7 dense-grid manual RDP bound meets epsilon cap",
    manual_dense_eps <= EPSILON_CAP,
    f"epsilon_dense={manual_dense_eps:.6f}; cap={EPSILON_CAP}"
)
record_check(
    "LOCAL-AUDIT-8 independent dp-accounting cross-check",
    DP_ACCOUNTING_AVAILABLE
    and np.isfinite(library_v3_eps) and library_v3_eps <= manual_v3_eps + 1e-7
    and np.isfinite(library_dense_eps) and library_dense_eps <= manual_dense_eps + 1e-6
    and library_v3_eps <= EPSILON_CAP + 1e-7
    and library_dense_eps <= EPSILON_CAP + 1e-6,
    f"available={DP_ACCOUNTING_AVAILABLE}; manual={manual_v3_eps:.6f}/{manual_dense_eps:.6f}; "
    f"library={library_v3_eps:.6f}/{library_dense_eps:.6f}"
)
record_check(
    "LOCAL-AUDIT-9 mean utility screen",
    utility_screen_pass,
    f"mean_DP_AP={dp_mean:.4f}; reference={utility_floor:.4f}"
)
record_check(
    "LOCAL-AUDIT-10 descriptive seed stability screen",
    stability_screen_pass,
    f"DP_APs={point_dp_aps.tolist()}; SD={dp_sd:.4f}; mean={dp_mean:.4f}"
)
record_check(
    "LOCAL-AUDIT-11 paired machine bootstrap completed",
    len(PAIRED) == 4 and len(boot_diff) > 0,
    f"B={BOOTSTRAP_B}; machines={n_clusters}; mean_diff={point_diff_mean:.4f}; CI=({boot_ci[0]:.4f},{boot_ci[1]:.4f})"
)
record_check(
    "LOCAL-AUDIT-12 selection lock records validation-only selection",
    lock.get("selection_split") == "validation only" and lock.get("test_loaded_before_selection") is False,
    "Checks protocol declaration; does not independently prove historical execution order."
)

all_critical_pass = all(v["status"] == "PASS" for v in checks.values())
if all_critical_pass and ci_above_zero:
    conclusion = (
        "The local-normalization utility screens and integrity/accounting checks pass. "
        "The paired machine-cluster bootstrap CI is above zero, indicating a benchmark-specific "
        "DP-vs-clipped AP advantage on this previously used test set. This is not proof of "
        "end-to-end DP or generalization to unseen factories."
    )
elif all_critical_pass:
    conclusion = (
        "Integrity, accounting, and utility screens pass, but the paired machine-bootstrap CI "
        "does not lie entirely above zero. Do not claim a statistically clear DP advantage "
        "over ClippedFedAvg from this audit."
    )
else:
    conclusion = (
        "The local-normalization audit is not closed. Resolve every failed critical check before "
        "finalizing Phase 3 results."
    )

report = {
    "title": "FedKG-Maint Phase 3 Local-Normalization Closure Audit v1",
    "source_run": "Phase 3 Privacy Utility LocalNorm v1",
    "normalization_policy": NORMALIZATION_POLICY,
    "test_set": "test_site_full / held-out factory_5",
    "warning": "test_site_full has been used in previous experiments; this is not an untouched external replication.",
    "n_test_rows": int(len(ytest)),
    "n_machines": int(n_clusters),
    "test_prevalence": prevalence,
    "selected_local_epochs": selected_local_epochs,
    "selected_clip_norm": selected_clip,
    "selected_noise_multiplier": selected_sigma,
    "rounds": ROUNDS,
    "delta": DELTA,
    "epsilon_cap": EPSILON_CAP,
    "epsilon_stored_in_lock": lock_epsilon,
    "epsilon_manual_v3_grid": manual_v3_eps,
    "epsilon_manual_dense_grid": manual_dense_eps,
    "epsilon_dp_accounting_v3_grid": library_v3_eps,
    "epsilon_dp_accounting_dense_grid": library_dense_eps,
    "dp_accounting_available": DP_ACCOUNTING_AVAILABLE,
    "dp_accounting_status": DP_ACCOUNTING_STATUS,
    "dp_accounting_version": DP_ACCOUNTING_VERSION,
    "per_seed_dp_AP": {str(seed): float(point_dp_aps[i]) for i, seed in enumerate(SEEDS)},
    "per_seed_clipped_AP": {str(seed): float(point_ctrl_aps[i]) for i, seed in enumerate(SEEDS)},
    "mean_dp_AP": dp_mean,
    "sd_dp_AP_sample": dp_sd,
    "mean_clipped_AP": control_mean,
    "mean_central_compact_AP": central_mean,
    "mean_paired_AP_difference_DP_minus_clipped": point_diff_mean,
    "paired_machine_bootstrap_ci_95_mean_difference": [float(boot_ci[0]), float(boot_ci[1])],
    "bootstrap_fraction_mean_difference_le_zero_descriptive": bootstrap_nonpositive_fraction,
    "bootstrap_replicates": BOOTSTRAP_B,
    "mean_utility_screen_pass": utility_screen_pass,
    "seed_stability_screen_pass": stability_screen_pass,
    "paired_ci_entirely_above_zero": ci_above_zero,
    "fedavg_vs_clipped_max_abs_score_difference_by_seed": baseline_prediction_differences,
    "all_critical_checks_pass": all_critical_pass,
    "checks": checks,
    "conclusion": conclusion,
    "privacy_scope": (
        "client-update mechanism only; add/remove-one client adjacency; trusted server; "
        "fixed denominator; full participation; no subsampling amplification; preprocessing "
        "statistics and validation-based model selection are outside this scoped claim"
    ),
    "limitations": [
        "This is a compact 7-feature linear model, not the 25-feature model.",
        "Four simulated training clients and one held-out factory are used.",
        "Factory_5 normalization uses earlier unlabeled feature-only calibration rows; this is unsupervised site adaptation, not zero-shot normalization.",
        "The benchmark test set was used in earlier experiments.",
        "Preprocessing simulates local partitions within one process and does not prove physical raw-data isolation.",
        "The epsilon claim is not end-to-end differential privacy.",
        "SecureAggSim remains arithmetic simulation, not cryptographic secure aggregation.",
        "Knowledge Graph and LLM requirements remain separate project components."
    ]
}

(AUDIT_DIR / "reports" / "phase3_localnorm_closure_audit_report.json").write_text(
    json.dumps(report, indent=2, default=str)
)

md = [
    "# FedKG-Maint — Phase 3 Local-Normalization Closure Audit v1",
    "",
    "> **Scope:** post-run audit of the local-normalization experiment. No training or tuning is performed. The existing held-out test set has been used in prior experiments.",
    "",
    "## Configuration and privacy accounting",
    "",
    f"- Normalization policy: `{NORMALIZATION_POLICY}`.",
    f"- Local epochs: `{selected_local_epochs}`; clipping norm: `{selected_clip}`; noise multiplier: `{selected_sigma}`; rounds: `{ROUNDS}`.",
    f"- Epsilon from the original manual RDP grid: `{manual_v3_eps:.6f}`.",
    f"- Manual dense-grid epsilon: `{manual_dense_eps:.6f}` at delta `{DELTA}`.",
    f"- `dp-accounting` epsilon: `{library_v3_eps:.6f}` on original orders; `{library_dense_eps:.6f}` on dense orders." if DP_ACCOUNTING_AVAILABLE else "- Independent `dp-accounting` result unavailable; manual RDP checks were calculated.",
    f"- `dp-accounting` status: `{DP_ACCOUNTING_STATUS}`; version: `{DP_ACCOUNTING_VERSION}`.",
    "- The epsilon claim covers the clipped/noised client-update channel only. It does not cover the complete data pipeline or system.",
    "",
    "## Recomputed final results",
    "",
    "| Method | Seeds | Mean PR-AUC | SD PR-AUC |",
    "|---|---:|---:|---:|"
]
for _, row in SUMMARY_RECALC.iterrows():
    md.append(f"| {row['method']} | {int(row['seeds'])} | {row['mean_test_AP']:.4f} | {row['sd_test_AP']:.4f} |")
md += [
    "",
    "## Primary paired comparison",
    "",
    f"- DP-FedAvg mean PR-AUC: `{dp_mean:.4f}` (sample SD `{dp_sd:.4f}`).",
    f"- ClippedFedAvg mean PR-AUC: `{control_mean:.4f}`.",
    f"- Mean paired difference (DP - clipped): `{point_diff_mean:.4f}`.",
    f"- 95% paired machine-cluster bootstrap CI: `[{boot_ci[0]:.4f}, {boot_ci[1]:.4f}]`.",
    f"- Descriptive fraction of bootstrap differences <= 0: `{bootstrap_nonpositive_fraction:.4f}` (not a formal p-value).",
    f"- Bootstrap replicates: `{BOOTSTRAP_B}`; held-out machine clusters: `{n_clusters}`.",
    "",
    "## Audit checks",
    ""
]
for name, item in checks.items():
    md.append(f"- **{item['status']}** — {name}: {item['detail']}")
md += [
    "",
    "## Decision",
    "",
    conclusion,
    "",
    "## Interpretation limits",
    ""
] + [f"- {item}" for item in report["limitations"]]
(AUDIT_DIR / "reports" / "phase3_localnorm_closure_audit_report.md").write_text("\n".join(md))

pd.DataFrame([{"check": name, **item} for name, item in checks.items()]).to_csv(
    AUDIT_DIR / "results" / "phase3_localnorm_closure_audit_checks.csv", index=False
)

print("\n" + conclusion)
print("Audit outputs:", AUDIT_DIR)
print("Paired comparison:", AUDIT_DIR / "results" / "paired_machine_bootstrap_dp_vs_clipped.csv")
